## ファイルの前処理

In [1]:
import os, requests
from collections import defaultdict
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [2]:
base_dir = "/home/shaeo/cadd_training/20260111_water_analysis"

In [3]:
# ファイルのダウンロード（1FJS: Factor Xa)
pdb_id = "1FJS"
pdb_id = pdb_id.lower()
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
os.makedirs(base_dir + "/data", exist_ok=True)
out_file = base_dir + f"/data/{pdb_id}.pdb"

response = requests.get(url)
if response.status_code == 200:
    with open(out_file, "wb") as f:
        f.write(response.content)
else:
    print(f"Failed to fetch {pdb_id} from OPM (status {response.status_code})")

In [4]:
# リガンドとレセプターの分割
protein_file = base_dir + f"/data/receptor_{pdb_id}.pdb"
ligand_file = base_dir + f"/data/ligand_{pdb_id}.pdb"
ligand_resname = "Z34"

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure("complex", out_file)
io = PDB.PDBIO()

# レセプターを抽出（チェインごとに「残す」残基範囲を指定）
class ProteinSelect(PDB.Select):
    def __init__(self, chain_include_ranges: dict[str, list[tuple[int, int]]]):
        self.chain_include_ranges = chain_include_ranges

    def accept_residue(self, residue):
        # 標準アミノ酸のみ
        if not PDB.is_aa(residue, standard=True):
            return False

        chain_id = residue.get_parent().id  # チェインID
        res_id = residue.id[1]              # residue number

        # 指定チェインでなければ除外
        if chain_id not in self.chain_include_ranges:
            return False

        # 指定された範囲に入っていれば残す
        for lo, hi in self.chain_include_ranges[chain_id]:
            if lo <= res_id <= hi:
                return True

        return False

include = {
    "A": [(16, 244)],
    "L": [(87, 52)],
}

io.set_structure(structure)
io.save(protein_file, ProteinSelect(include))

# リガンドを抽出
class LigandSelect(PDB.Select):
    def accept_residue(self, residue):
        return residue.get_resname().strip() == ligand_resname
io.set_structure(structure)
io.save(ligand_file, LigandSelect())

In [5]:
# リガンドのファイル形式を変換: PyMOLでリガンドを抽出
ligand_sdf = base_dir + f"/data/ligand_{pdb_id}.sdf"
!obabel $ligand_sdf -O $ligand_sdf -ph 7.4

1 molecule converted


In [6]:
protein_file2 = base_dir + f"/data/receptor_{pdb_id}_del.pdb"

def strip_icode_auto_lines(pdb_lines):
    has_blank = defaultdict(bool)

    # 1st pass: 無印が存在する残基を記録
    for l in pdb_lines:
        if l.startswith(("ATOM  ", "HETATM")) and len(l) >= 27:
            key = (l[21], l[22:26].strip())  # (chain, resSeq)
            if l[26] == " ":
                has_blank[key] = True

    # 2nd pass: 無印がある残基では iCode 付き行を除去
    out = []
    for l in pdb_lines:
        if l.startswith(("ATOM  ", "HETATM")) and len(l) >= 27:
            key = (l[21], l[22:26].strip())
            if has_blank.get(key, False) and l[26] != " ":
                continue
        out.append(l)

    return out

with open(protein_file, mode="r") as f:
    lines = f.readlines()

cleaned = strip_icode_auto_lines(lines)

with open(protein_file2, mode="w") as f:
    f.writelines(cleaned)

In [7]:
# PDBfixer
protein_file_fixed = base_dir + f"/data/receptor_{pdb_id}-fixed.pdb"

fixer = PDBFixer(filename=protein_file2)
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()

with open(protein_file_fixed, "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)

In [8]:
# Amber用にファイルを変換
protein_file_amber = base_dir + f"/data/receptor_{pdb_id}_amber.pdb"

%cd {base_dir}/data
%mkdir temp
%cd temp
!pdb4amber -i {protein_file_fixed} -o {protein_file_amber} --nohyd
%cd {base_dir}/data
!rm -r {base_dir}/data/temp

/home/shaeo/cadd_training/20260111_water_analysis/data


/home/shaeo/cadd_training/20260111_water_analysis/data/temp

Summary of pdb4amber for: /home/shaeo/cadd_training/20260111_water_analysis/data/receptor_1fjs-fixed.pdb

----------Chains
The following (original) chains have been found:
A

---------- Alternate Locations (Original Residues!))

The following residues had alternate locations:
None
-----------Non-standard-resnames


---------- Gaps (Renumbered Residues!)
gap of 4.607921 A between GLN 46 and LYS 47
gap of 4.074366 A between PRO 109 and ARG 110
gap of 3.939615 A between THR 116 and THR 117
gap of 4.605524 A between TYR 169 and LYS 170
gap of 3.093242 A between LYS 206 and LYS 207

---------- Missing heavy atom(s)

None
/home/shaeo/cadd_training/20260111_water_analysis/data


## 複合体の準備

In [9]:
import subprocess, textwrap

In [10]:
# リガンドのパラメータを作成
ligand_frcmod = base_dir + f"/data/ligand_{pdb_id}.frcmod"
ligand_mol2 = base_dir + f"/data/ligand_{pdb_id}.mol2"

%cd {base_dir}/data
%mkdir temp
%cd temp
!antechamber -i {ligand_sdf} -fi sdf -o {ligand_mol2} -fo mol2 -c bcc -nc 1 -at gaff2
!parmchk2    -i {ligand_mol2} -f mol2 -o {ligand_frcmod}
%cd {base_dir}/data
!rm -r {base_dir}/data/temp

/home/shaeo/cadd_training/20260111_water_analysis/data


/home/shaeo/cadd_training/20260111_water_analysis/data/temp
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to gaff2; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for sdf File --
   Status: pass
Info: Bond types are assigned for valence state (1) with penalty (1).
Info: Total number of electrons: 272; net charge: 1

Running: /home/shaeo/miniconda3/envs/openmm/bin/sqm -O -i sqm.in -o sqm.out

/home/shaeo/cadd_training/20260111_water_analysis/data


In [11]:
# tleap.inを作成
parm7_file = base_dir + f"/data/system_{pdb_id}.parm7"
rst7_file = base_dir + f"/data/system_{pdb_id}.rst7"
system_file = base_dir + f"/data/system_{pdb_id}.pdb"
leap_in_path = base_dir + "/data/tleap.in"

tleap_in = f"""
source leaprc.protein.ff14SB
source leaprc.water.tip3p
source leaprc.gaff2

# ligand パラメータ
loadamberparams {ligand_frcmod}
LIG = loadmol2 {ligand_mol2}

# protein
PROT = loadpdb {protein_file_amber}

# 複合体
COMPLEX = combine {{ PROT LIG }}

# 溶媒化
solvatebox COMPLEX TIP3PBOX 10.0

# イオン追加
addions COMPLEX Na+ 0
addions COMPLEX Cl- 0

# 出力
saveamberparm COMPLEX {parm7_file} {rst7_file}
savepdb COMPLEX {system_file}

quit
"""

with open(leap_in_path, mode="w") as f:
    f.write(textwrap.dedent(tleap_in))

In [12]:
# tleap実行（ログ保存）
leap_log_path = base_dir + "/data/leap.log"
cmd = ["tleap", "-f", leap_in_path]
res = subprocess.run(cmd, capture_output=True, text=True)
with open(leap_log_path, "w") as f:
    f.write(res.stdout)
    f.write("\n\n[STDERR]\n")
    f.write(res.stderr)

print("tleap return code:", res.returncode)
print("leap.in:", leap_in_path)
print("leap.log:", leap_log_path)
print("parm7 exists:", os.path.exists(parm7_file))
print("rst7 exists:", os.path.exists(rst7_file))

# 失敗時はログのERROR行だけ抜く
if res.returncode != 0 or (not os.path.exists(parm7_file)) or (not os.path.exists(rst7_file)):
    print("\n---- tleap errors (grep) ----")
    for line in (res.stdout + "\n" + res.stderr).splitlines():
        if "ERROR" in line.upper() or "FATAL" in line.upper() or "Could not" in line:
            print(line)
    raise RuntimeError("tleap failed; check leap.log")

tleap return code: 0
leap.in: /home/shaeo/cadd_training/20260111_water_analysis/data/tleap.in
leap.log: /home/shaeo/cadd_training/20260111_water_analysis/data/leap.log
parm7 exists: True
rst7 exists: True


## MDシミュレーション

In [13]:
import mdtraj as md
from openmm import *
from openmm.app import *
from openmm.unit import *
import parmed as pmd

In [14]:
result_dir = base_dir + "/result"
os.makedirs(result_dir, exist_ok=True)

In [15]:
# simulationオブジェクトを作成

# Amberファイル読み込み
prmtop = AmberPrmtopFile(parm7_file)
inpcrd = AmberInpcrdFile(rst7_file)

# システム
system = prmtop.createSystem(
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometer,
    constraints=HBonds
)

# バロスタット
barostat = MonteCarloBarostat(
    1*bar,        # 圧力
    10*kelvin,   # 温度（シミュレーション温度に合わせる）
    25                 # 試行頻度
)
system.addForce(barostat)

# インテグレーター
integrator = LangevinIntegrator(
    10*kelvin,
    1/picosecond,
    0.002*picoseconds
)

# プラットフォーム
platform = Platform.getPlatformByName('CUDA')  # または 'OpenCL', 'CPU'
properties = {'Precision': 'mixed'} # GPUの計算精度

# トポロジーと座標の取得
topology = prmtop.topology
positions = inpcrd.positions

simulation = Simulation(topology, system, integrator, platform, properties)
simulation.context.setPositions(positions)

if inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

### 1.エネルギー最適化

In [16]:
em_path = result_dir + '/1_minimization'
os.makedirs(em_path, exist_ok=True)

In [17]:
state_path_em = em_path + "/state.xml"
structure_path_em = em_path + "/structure.pdb"

In [18]:
# 実行
simulation.minimizeEnergy()

In [19]:
# 保存
state_em = simulation.context.getState(
    getPositions=True, 
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_em, "w") as f:
    f.write(XmlSerializer.serialize(state_em))
positions_em = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_em, "w") as f:
    PDBFile.writeFile(simulation.topology, positions_em, f)

### 2.等温緩和

In [20]:
qd_path = result_dir + '/2_quenched_dynamics'
os.makedirs(qd_path, exist_ok=True)

In [21]:
state_path_qd = qd_path + "/state.xml"
structure_path_qd = qd_path + "/structure.pdb"
dcd_path_qd = qd_path + "/traj.dcd"
log_path_qd = qd_path + "/log.log"

In [22]:
# 設定
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(DCDReporter(dcd_path_qd, 2))
simulation.reporters.append(
    StateDataReporter(
        log_path_qd, 2, totalSteps=10, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)
simulation.context.setState(state_em)

In [23]:
# 実行
simulation.step(10)

In [24]:
# 保存
state_qd = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_qd, "w") as f:
    f.write(XmlSerializer.serialize(state_qd))
positions_qd = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_qd, "w") as f:
    PDBFile.writeFile(simulation.topology, positions_qd, f)

### 3.昇温

In [25]:
ht_path = result_dir + '/3_heating'
os.makedirs(ht_path, exist_ok=True)

In [26]:
state_path_ht = ht_path + "/state.xml"
structure_path_ht = ht_path + "/structure.pdb"
dcd_path_ht = ht_path + "/traj.dcd"
log_path_ht = ht_path + "/log.log"

In [27]:
# 設定
n_steps = 250000  # 2fs * 250,000 = 500,000fs ← 500ps ← 0.5ns
simulation.context.setState(state_qd)
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(DCDReporter(dcd_path_ht, 5000))
simulation.reporters.append(
    StateDataReporter(
        log_path_ht, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [28]:
# 拘束条件: タンパク主鎖、リガンド
restraint = CustomExternalForce("k*periodicdistance(x, y, z, x0, y0, z0)^2")
res_idx = system.addForce(restraint)
restraint.addGlobalParameter("k", 1000.0*kilojoules_per_mole/nanometer)
restraint.addPerParticleParameter("x0")
restraint.addPerParticleParameter("y0")
restraint.addPerParticleParameter("z0")

protein_atoms = [atom for atom in topology.atoms() if atom.name in ["CA", "N", "C", "O"]]
for atom in protein_atoms:
   restraint.addParticle(atom.index, positions[atom.index])
ligand_atoms = [atom for atom in topology.atoms() if atom.residue.name=="MOL"]
for atom in protein_atoms:
   restraint.addParticle(atom.index, positions[atom.index])

In [29]:
# 実行
list_temp = [100, 150, 200, 250, 300]
list_step = [50000, 50000, 50000, 50000, 50000]
for temp, step in zip(list_temp, list_step):
    integrator.setTemperature(temp*kelvin)
    barostat.setDefaultTemperature(temp*kelvin)
    simulation.step(step)

In [30]:
# 保存
state_ht = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_ht, "w") as f:
    f.write(XmlSerializer.serialize(state_ht))
positions_ht = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_ht, "w") as f:
    PDBFile.writeFile(simulation.topology, positions_ht, f)

### 4.平衡化

In [31]:
eq_path = result_dir + '/4_equibaration'
os.makedirs(eq_path, exist_ok=True)

In [32]:
state_path_eq = eq_path + "/state.xml"
structure_path_eq = eq_path + "/structure.pdb"
dcd_path_eq = eq_path + "/traj.dcd"
log_path_eq = eq_path + "/log.log"

In [33]:
# 設定
n_steps = 250000  # 2fs * 250,000 = 500,000fs ← 500ps ← 0.5ns
simulation.context.setState(state_ht)
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(DCDReporter(dcd_path_eq, 5000))
simulation.reporters.append(
    StateDataReporter(
        log_path_eq, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [34]:
# 実行
list_restrain = [1000.0, 300.0, 100.0, 30.0, 0.0]
list_step = [5000, 40000, 40000, 40000, 125000]
for k, step in zip(list_restrain, list_step):
    restraint.setGlobalParameterDefaultValue(0, k*kilojoules_per_mole/nanometer)  
    simulation.step(step)

In [35]:
# 保存
state_eq = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_eq, "w") as f:
    f.write(XmlSerializer.serialize(state_eq))
positions_eq = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_eq, "w") as f:
    PDBFile.writeFile(simulation.topology, positions_eq, f)

### 5.生産

In [36]:
md_path = result_dir + '/5_production'
os.makedirs(md_path, exist_ok=True)

In [37]:
state_path_md = md_path + "/state.xml"
structure_path_md = md_path + "/structure.pdb"
dcd_path_md = md_path + "/traj.dcd"
log_path_md = md_path + "/log.log"
checkpoint_path = md_path + "/checkpoint.chk"

In [38]:
# 設定
n_steps = 500000  # 2fs * 500,000 = 1,000,000fs ← 1000ps ← 1.0ns
simulation.context.setState(state_eq)
simulation.currentStep = 0
simulation.reporters = []
simulation.reporters.append(DCDReporter(dcd_path_md, 5000))
simulation.reporters.append(
    StateDataReporter(
        log_path_md, 5000, totalSteps=n_steps, step=True, speed=True, progress=True, potentialEnergy=True, temperature=True
    )
)

In [39]:
# 実行
simulation.step(n_steps)

In [40]:
# 保存
state_md = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True,
    getForces=True,
    getParameters=True,
    enforcePeriodicBox=True
)
with open(state_path_md, "w") as f:
    f.write(XmlSerializer.serialize(state_md))
positions_md = simulation.context.getState(getPositions=True).getPositions()
with open(structure_path_md, "w") as f:
    PDBFile.writeFile(simulation.topology, positions_md, f)
with open(checkpoint_path, "wb") as f:
    f.write(simulation.context.createCheckpoint())

### 6.トラジェクトリの後処理

In [41]:
fixed_path_ht = eq_path + "/traj_pc.dcd"
fixed_path_eq = ht_path + "/traj_pc.dcd"
fixed_path_md = md_path + "/traj_pc.dcd"

In [42]:
dict_path_dcd = {
    dcd_path_ht:fixed_path_ht,
    dcd_path_eq:fixed_path_eq,
    dcd_path_md:fixed_path_md,
}

In [43]:
for dcd_path, fixed_dcd_path in dict_path_dcd.items():
    traj = md.load(dcd_path, top=structure_path_em)
    traj_whole = traj.make_molecules_whole()    # 分子を “ちぎれない” 形に直す（make whole）
    prot = traj_whole.topology.select("protein")    # タンパクを基準にセンタリング＆ボックス内にラップ
    traj_centered = traj_whole.center_coordinates()
    traj_wrapped = traj_centered.image_molecules()  # 近接イメージに押し戻す
    traj_fit = traj_wrapped.superpose(traj_wrapped, frame=0, atom_indices=prot)     # 剛体合わせ（平行移動・回転除去）してRMSD等が滑らかに
    traj_fit.save_dcd(fixed_dcd_path)

In [132]:
# .dcd → .nc
dcd2nc_path = md_path + "/dcd2nc.in"
nc_path_md = md_path + "/traj_pc.nc"

dcd2nc = f"""
parm {parm7_file}
trajin {fixed_path_md}
trajout {nc_path_md} netcdf
run
"""

with open(dcd2nc_path, mode="w") as f:
    f.write(textwrap.dedent(dcd2nc))

%cd {md_path}
!cpptraj -i {dcd2nc_path} 

/home/shaeo/cadd_training/20260111_water_analysis/result/5_production

CPPTRAJ: Trajectory Analysis. V6.24.0 (AmberTools)
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| Date/time: 01/11/26 19:55:21
| Available memory: 18.924 GB

INPUT: Reading input from '/home/shaeo/cadd_training/20260111_water_analysis/result/5_production/dcd2nc.in'
  [parm /home/shaeo/cadd_training/20260111_water_analysis/data/system_1fjs.parm7]
	Reading '/home/shaeo/cadd_training/20260111_water_analysis/data/system_1fjs.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /home/shaeo/cadd_training/20260111_water_analysis/result/5_production/traj_pc.dcd]
	Reading '/home/shaeo/cadd_training/20260111_water_analysis/result/5_production/traj_pc.dcd' as Charmm DCD
	Symmetric shape matrix detected.
  [trajout /home/shaeo/cadd_training/20260111_water_analysis/result/5_production/traj_pc.nc netcdf]
	Writing '/home/shaeo/cadd_training/20260111_water_analysis/result/5_producti

## 解析

In [83]:
import copy
import pytraj as pt

### GIST

In [124]:
gist_dir = base_dir + "/GIST"
os.makedirs(gist_dir, exist_ok=True)

In [139]:
gistpp_dir = "/home/shaeo/opt/gist-post-processing"

In [125]:
# リガンドの中心座標を算出
traj = pt.load(nc_path_md, parm7_file)
mask = traj.top.select(':MOL')

coords = traj.xyz[0][mask]
center = coords.mean(axis=0)

X, Y, Z = center
print(X, Y, Z)

20.162971168267923 27.36777430675069 29.299876322511768


In [126]:
# gist.inを作成
gist_in_file = gist_dir + "/gist.in"

gist_in = f"""
parm {parm7_file}
trajin {nc_path_md}

gist gridcntr {X:.6f} {Y:.6f} {Z:.6f} griddim 60 60 60 gridspacn 0.5 temp 300.0 out gist
go
"""

with open(gist_in_file, mode="w") as f:
    f.write(textwrap.dedent(gist_in))

In [137]:
# GIST
%cd {gist_dir}
!cpptraj -i {gist_in_file}
%cd {base_dir}

/home/shaeo/cadd_training/20260111_water_analysis/GIST

CPPTRAJ: Trajectory Analysis. V6.24.0 (AmberTools)
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| Date/time: 01/11/26 19:59:31
| Available memory: 18.829 GB

INPUT: Reading input from '/home/shaeo/cadd_training/20260111_water_analysis/GIST/gist.in'
  [parm /home/shaeo/cadd_training/20260111_water_analysis/data/system_1fjs.parm7]
	Reading '/home/shaeo/cadd_training/20260111_water_analysis/data/system_1fjs.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /home/shaeo/cadd_training/20260111_water_analysis/result/5_production/traj_pc.nc]
	Reading '/home/shaeo/cadd_training/20260111_water_analysis/result/5_production/traj_pc.nc' as Amber NetCDF
  [gist gridcntr 20.162971 27.367774 29.299876 griddim 60 60 60 gridspacn 0.5 temp 300.0 out gist]
    GIST:
	Output prefix= 'gist', grid output extension= '.dx'
	Output float format string= '%g', output integer format string= '%i'
	GIST info 

In [140]:
# 総溶媒エネルギーを算出
dx_Esw_dense = gist_dir + "/gist-Esw-dens.dx"
dx_hald_Esw_dense = gist_dir + "/half-gist-Esw-dens.dx"
dx_Eww_dense = gist_dir + "/gist-Eww-dens.dx"
dx_Etot_dense = gist_dir + "/gist-Etot-dens.dx"
dx_Etot = gist_dir + "/gist-Etot.dx"

%cd {gistpp_dir}
!./gistpp -i {dx_Esw_dense} -op multconst -opt const 0.5 -o {dx_hald_Esw_dense}
!./gistpp -i {dx_Eww_dense} -i2 {dx_hald_Esw_dense} -op add -o {dx_Etot_dense}
!./gistpp -i {dx_Etot_dense} -op multconst -opt const 0.125 -o {dx_Etot}
%cd {base_dir}

/home/shaeo/opt/gist-post-processing
Voxels in: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Esw-dens.dx to have constant 0.5 multiplied to to them, then new values written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/half-gist-Esw-dens.dx
/home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Eww-dens.dx to be added to /home/shaeo/cadd_training/20260111_water_analysis/GIST/half-gist-Esw-dens.dx and written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Etot-dens.dx
Voxels in: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Etot-dens.dx to have constant 0.125 multiplied to to them, then new values written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Etot.dx
/home/shaeo/cadd_training/20260111_water_analysis


In [141]:
# エントロピーを集約
dx_dTStrans_dense = gist_dir + "/gist-dTStrans-dens.dx"
dx_dTSorient_dense = gist_dir + "/gist-dTSorient-dens.dx"
dx_dTStot_dense = gist_dir + "/gist-dTStot_dense.dx"
dx_dTStot = gist_dir + "/gist-dTStot.dx"

%cd {gistpp_dir}
!./gistpp -i {dx_dTStrans_dense} -i2 {dx_dTSorient_dense} -op add -o {dx_dTStot_dense}
!./gistpp -i {dx_dTStot_dense} -op multconst -opt const 0.125 -o {dx_dTStot}
%cd {base_dir}

/home/shaeo/opt/gist-post-processing
/home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTStrans-dens.dx to be added to /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTSorient-dens.dx and written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTStot_dense.dx
Voxels in: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTStot_dense.dx to have constant 0.125 multiplied to to them, then new values written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTStot.dx
/home/shaeo/cadd_training/20260111_water_analysis


In [142]:
# 溶媒自由エネルギーを算出
dx_dA = gist_dir + "/gist-dA.dx"

%cd {gistpp_dir}
!./gistpp -i {dx_Etot} -i2 {dx_dTStot} -op sub -o {dx_dA}
%cd {base_dir}

/home/shaeo/opt/gist-post-processing
/home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dTStot.dx to be subtracted from /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-Etot.dx and written to: /home/shaeo/cadd_training/20260111_water_analysis/GIST/gist-dA.dx
/home/shaeo/cadd_training/20260111_water_analysis


### 3D-RISM ※できなかった。conda非推奨らしい

In [47]:
rims_dir = base_dir + "/3D-RISM"
os.makedirs(rims_dir, exist_ok=True)

In [ ]:
CONDA_PREFIX = os.environ["CONDA_PREFIX"]

In [66]:
# 複合体のファイルを作成
parm7_complex_file = base_dir + "/data/complex_1fjs.parm7"
rst7_complex_file = base_dir + "/data/complex_1fjs.rst7"
pdb_complex_file = base_dir + "/data/complex_1fjs.pdb"

# parm7 + rst7 を読み込み
structure = pmd.load_file(
    parm7_file,
    rst7_file
)

# コピーを作成
dry = copy.deepcopy(structure)

# 水・イオンを除去
dry.strip(":WAT,Na+,Cl-")

# 保存
dry.save(parm7_complex_file, overwrite=True)
dry.save(rst7_complex_file, overwrite=True)

!ambpdb -p {parm7_complex_file} -c {rst7_complex_file} > {pdb_complex_file}

In [49]:
# 溶媒の Xvv を作る（1D-RISM）
rism1d_inp_path = rims_dir + "/rism1d.inp"
rism1d_log_path = rims_dir + "/rism1d.log"

rism1d_inp = f"""
&PARAMETERS
OUTLIST='gx', THEORY='DRISM', CLOSURE='PSE4',
!grid
NR=16384, DR=0.025,
!MDIIS
MDIIS_NVEC=20, MDIIS_DEL=0.3, TOLERANCE=1.e-12,
!iter
KSAVE=-1, maxstep=10000,
!bulk solvent properties
TEMPERATURE=298.15, DIEPS=78.44
NSP=3
/
&SPECIES
!cSPCE water
units = 'M' !unit in mol/L
DENSITY = 55.5D0,
!if you have the file in the .inp directory
MODEL = "cSPCE.mdl"
!if loading from amber directory
MODEL="{CONDA_PREFIX}/dat/rism1d/mdl/cSPCE.mdl"
/
&SPECIES
!Sodium
units = 'M'
DENSITY = 0.1D0,
!if you have the file in the .inp directory
MODEL = "Na+.mdl"
!if loading from amber directory
MODEL="{CONDA_PREFIX}/dat/rism1d/mdl/jc_SPCE/Na+.mdl"
/
&SPECIES
!Chloride
units = 'M'
DENSITY = 0.1d0,
!if you have the file in the .inp directory
MODEL = "Cl-.mdl"
!if loading from amber directory
MODEL = "{CONDA_PREFIX}/dat/rism1d/mdl/jc_SPCE/Cl-.mdl"
/
"""

with open(rism1d_inp_path, mode="w") as f:
    f.write(textwrap.dedent(rism1d_inp))
    
%cd {rims_dir}
!rism1d {rims_dir}/rism1d > {rism1d_log_path}
%cd {base_dir}

/home/shaeo/cadd_training/20260111_water_analysis/3D-RISM
/home/shaeo/cadd_training/20260111_water_analysis


In [80]:
# 3D-RISMを実行
rism3d_out_path = rims_dir + f"/rism3d.out"

%cd {rims_dir}
!rism3d.snglpnt \
    --pdb {pdb_complex_file} \
    --prmtop {parm7_complex_file} \
    --rst {rst7_complex_file} \
    --xvv rism1d.xvv \
    --closure kh \
    --solvbox 160,160,160 \
    --ng 160,160,160 \
    --guv g \
    --volfmt dx \
    --verbose 2 --progress \
    > {rism3d_out_path} 2> {rism3d_out_path}.err
%cd {base_dir}

/home/shaeo/cadd_training/20260111_water_analysis/3D-RISM
/home/shaeo/cadd_training/20260111_water_analysis
